# Arabic Text Augmentation for Low-Resource Classification

This notebook compares Back-Translation, masked language modeling (MLM), and Controlled Generation with MARBERT on ASTD. It shares one configuration, one data split, and one training routine.

Run from the project checkout or point `ASTD_PROJECT_ROOT` to the extracted project. On Colab, the first cell discovers a ZIP already uploaded to the Files panel. A CUDA runtime and project dependencies are required for model execution. Install transformers>=4.44,<5, PyTorch with CUDA, numpy, pandas, sentencepiece, sacremoses and scikit-learn into the notebook kernel before running the model stages. Candidate review is recorded per row in each generated review CSV. After a review, rerun the notebook to train on approved rows. Existing candidate files and completed metrics are reused.

## 1. Locate data and configure the experiment

Find the ASTD project, verify the required split files and detect the dependencies and CUDA device. No model is downloaded at this stage.

In [1]:
import ast
import gc
import csv
import importlib.util
import json
import os
import random
import re
import zipfile
from pathlib import Path

SEED = 42
FRACTIONS = [10, 25, 50]
LABELS = ['OBJ', 'NEG', 'NEUTRAL', 'POS']
METHODS = ('back_translation', 'mlm', 'controlled_generation')
METHOD = 'controlled_generation'
GENERATOR_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SOURCES_PER_CLASS = None
ROOT = Path(os.environ.get('ASTD_PROJECT_ROOT', Path.cwd())).resolve()
root_marker = Path('data/processed/astd_train.csv')
if not (ROOT / root_marker).is_file():
    candidates = [ROOT.parent, ROOT / 'arabic-text-augmentation-low-resource-main']
    if Path('/content').is_dir():
        runtime = Path('/content/project')
        candidates += sorted(p.parents[2] for p in runtime.rglob(str(root_marker)))
        if not any((p / root_marker).is_file() for p in candidates):
            archives = [Path('/content') / name for name in (
                'student2_project_ready.zip', 'arabic-text-augmentation-low-resource-main.zip')]
            archive_path = next((p for p in archives if p.is_file()), None)
            if archive_path:
                with zipfile.ZipFile(archive_path) as z:
                    if any(not (runtime / n).resolve().is_relative_to(runtime.resolve())
                           for n in z.namelist()):
                        raise ValueError('Archive contains an unsafe path')
                    z.extractall(runtime)
                candidates += sorted(p.parents[2] for p in runtime.rglob(str(root_marker)))
    ROOT = next((p for p in candidates if (p / root_marker).is_file()), ROOT)

SPLIT_DIR = ROOT / 'data/processed/low_resource_splits_v2'
OUT = ROOT / 'reports/augmentation'
AUG = ROOT / 'data/processed/augmentation'
data_paths = [ROOT / root_marker, ROOT / 'data/processed/astd_validation.csv',
              ROOT / 'data/processed/astd_test.csv']
data_paths += [SPLIT_DIR / f'train_{pct}pct.csv' for pct in FRACTIONS]
missing_data = [str(p.relative_to(ROOT)) for p in data_paths if not p.is_file()]
packages = ('torch', 'numpy', 'pandas', 'transformers', 'sentencepiece', 'sacremoses', 'sklearn')
missing_packages = [p for p in packages if importlib.util.find_spec(p) is None]
if 'torch' not in missing_packages:
    import torch
    cuda = torch.cuda.is_available()
else:
    cuda = False
CAN_RUN = cuda and not missing_data and not missing_packages
print('Missing dataset files:', missing_data or 'none')
if not missing_data:
    for pct in FRACTIONS:
        path = SPLIT_DIR / f'train_{pct}pct.csv'
        with path.open(encoding='utf-8-sig', newline='') as stream:
            rows = list(csv.DictReader(stream))
        assert {row['label'] for row in rows} == set(LABELS)
        print(f'{pct}% split: {len(rows)} rows, four labels verified.')
print('Missing packages:', missing_packages or 'none')
print('CUDA available:', cuda)
if CAN_RUN:
    import numpy as np
    import pandas as pd
    from IPython.display import display
    OUT.mkdir(parents=True, exist_ok=True)
    AUG.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)


Missing dataset files: none
10% split: 600 rows, four labels verified.
25% split: 1501 rows, four labels verified.
50% split: 3002 rows, four labels verified.
Missing packages: ['torch', 'transformers', 'sentencepiece', 'sacremoses']
CUDA available: False


## 2. Define candidate controls

Define Arabic label instructions, normalize text and check the candidate filter on four synthetic strings. This check does not train a model.

In [2]:

LABEL_GUIDES = {
    'OBJ': 'وصف موضوعي أو معلومة بلا حكم شخصي',
    'NEG': 'رأي سلبي واضح أو شكوى أو انتقاد',
    'NEUTRAL': 'رأي مختلط يجمع بين جانب إيجابي وجانب سلبي',
    'POS': 'رأي إيجابي واضح أو ثناء',
}

def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

def candidate(text, excluded):
    text = normalize_text(text).strip('"«»')
    if len(text) < 5 or len(text) > 280 or text in excluded:
        return None
    if not re.search(r'[ء-ي]', text) or re.search(r'https?://|www\.|@\w+', text):
        return None
    return text

assert candidate('التجربة ممتازة وسأكررها', set())
assert candidate('التجربة ممتازة وسأكررها', {'التجربة ممتازة وسأكررها'}) is None
assert candidate('English only', set()) is None
assert candidate('التجربة ممتازة https://example.org', set()) is None
print('Four filter smoke checks passed with synthetic inputs.')


Four filter smoke checks passed with synthetic inputs.


## 3. Generate Back-Translation candidates

Translate Arabic training examples through English and back to Arabic, retaining the source label and row identifier.

In [3]:
if CAN_RUN:
    pending = [pct for pct in FRACTIONS if not (AUG / f'back_translation_{pct}pct_review.csv').is_file()
               and not (AUG / f'back_translation_{pct}pct_raw.csv').is_file()]
    if not pending:
        print('Reusing existing back_translation candidate files.')
    if pending:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        AR_EN = 'Helsinki-NLP/opus-mt-ar-en'
        EN_AR = 'Helsinki-NLP/opus-mt-en-ar'
        def load_translator(model_id):
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(DEVICE).eval()
            @torch.inference_mode()
            def translate(texts, batch_size=8, max_length=256, truncation=True):
                results = []
                for start in range(0, len(texts), batch_size):
                    inputs = tokenizer(
                        texts[start:start + batch_size], return_tensors='pt',
                        padding=True, truncation=truncation, max_length=max_length,
                    ).to(DEVICE)
                    output = model.generate(**inputs, max_length=max_length)
                    results.extend(
                        {'translation_text': value}
                        for value in tokenizer.batch_decode(output, skip_special_tokens=True)
                    )
                return results
            return translate
        translator_forward = load_translator(AR_EN)
        translator_backward = load_translator(EN_AR)
        print('Translation models loaded:', AR_EN, EN_AR, '|', DEVICE)
        def normalize_text(x): return re.sub(r'\s+',' ',str(x)).strip()
        def translate_batch(texts, batch_size=8):
            english=translator_forward(texts,batch_size=batch_size,max_length=256,truncation=True)
            back=translator_backward([r['translation_text'] for r in english],batch_size=batch_size,max_length=256,truncation=True)
            return [normalize_text(r['translation_text']) for r in back]
        def make_backtranslation(df, pct):
            src=df if MAX_SOURCES_PER_CLASS is None else df.groupby('label',group_keys=False).head(MAX_SOURCES_PER_CLASS)
            records=[]
            for start in range(0,len(src),8):
                chunk=src.iloc[start:start+8]; texts=chunk.text.astype(str).tolist()
                try: outputs=translate_batch(texts)
                except Exception as exc:
                    print('Translation failed for batch',start,repr(exc));continue
                for (_,row),out in zip(chunk.iterrows(),outputs):
                    records.append({'source_row_id':row.row_id,'source_text':row.text,'text':out,'label':row.label,'label_id':row.label_id,'method':'back_translation','fraction':pct})
            return pd.DataFrame(records)

        for pct in pending:
            base = pd.read_csv(SPLIT_DIR / f'train_{pct}pct.csv')
            generated = make_backtranslation(base, pct)
            if generated.empty:
                print(f'No candidates produced at {pct}%.')
                continue
            output_path = AUG / f'back_translation_{pct}pct_raw.csv'
            temp = output_path.with_suffix('.tmp')
            generated.to_csv(temp, index=False)
            temp.replace(output_path)
            print(f'{pct}%: {len(generated)} candidates')
        del translator_forward, translator_backward
        gc.collect()
        torch.cuda.empty_cache()
else:
    print('Skipped: required runtime, data, or reviewed candidates are unavailable.')


Skipped: required runtime, data, or reviewed candidates are unavailable.


## 4. Generate MLM candidates

Mask one eligible non-negation token and substitute a contextually probable Arabic token. Save generated training candidates by fraction.

In [4]:
if CAN_RUN:
    pending = [pct for pct in FRACTIONS if not (AUG / f'mlm_{pct}pct_review.csv').is_file()
               and not (AUG / f'mlm_{pct}pct_raw.csv').is_file()]
    if not pending:
        print('Reusing existing mlm candidate files.')
    if pending:
        from transformers import AutoTokenizer, AutoModelForMaskedLM
        MLM_MODEL='aubmindlab/bert-base-arabertv02'
        mlm_tokenizer=AutoTokenizer.from_pretrained(MLM_MODEL)
        mlm_model=AutoModelForMaskedLM.from_pretrained(MLM_MODEL).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
        print('MLM loaded:',MLM_MODEL)
        NEGATION={'لا','لم','لن','ليس','ما','مو','مش','غير'}
        def mlm_one(text, seed=42):
            rng=random.Random(seed)
            enc=mlm_tokenizer(str(text),return_tensors='pt',truncation=True,max_length=128)
            ids=enc['input_ids'][0].clone()
            special=set(mlm_tokenizer.all_special_ids)
            positions=[]
            for i,tid in enumerate(ids.tolist()):
                token=mlm_tokenizer.convert_ids_to_tokens(int(tid))
                if tid not in special and not token.startswith('##') and token not in NEGATION:
                    positions.append(i)
            if not positions: return ''
            pos=rng.choice(positions); original=int(ids[pos]); ids[pos]=mlm_tokenizer.mask_token_id
            inputs={k:v.to(mlm_model.device) for k,v in enc.items()}
            inputs['input_ids'][0,pos]=mlm_tokenizer.mask_token_id
            with torch.inference_mode(): logits=mlm_model(**inputs).logits[0,pos]
            for candidate in torch.topk(logits, k=min(30,logits.numel())).indices.tolist():
                if candidate==original or candidate in special: continue
                token=mlm_tokenizer.convert_ids_to_tokens(candidate)
                if token.startswith('##') or token in NEGATION:continue
                ids[pos]=candidate
                output=normalize_text(mlm_tokenizer.decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=True))
                if output!=normalize_text(text):return output
            return ''
        def make_mlm(df,pct):
            src=df if MAX_SOURCES_PER_CLASS is None else df.groupby('label',group_keys=False).head(MAX_SOURCES_PER_CLASS)
            rows=[]
            for i,row in enumerate(src.itertuples(index=False)):
                try: new=mlm_one(row.text,seed=SEED+i)
                except Exception as exc:
                    print('MLM failed',row.row_id,repr(exc));continue
                rows.append({'source_row_id':row.row_id,'source_text':row.text,'text':new,'label':row.label,'label_id':row.label_id,'method':'mlm','fraction':pct})
            return pd.DataFrame(rows)

        for pct in pending:
            base = pd.read_csv(SPLIT_DIR / f'train_{pct}pct.csv')
            generated = make_mlm(base, pct)
            if generated.empty:
                print(f'No candidates produced at {pct}%.')
                continue
            output_path = AUG / f'mlm_{pct}pct_raw.csv'
            temp = output_path.with_suffix('.tmp')
            generated.to_csv(temp, index=False)
            temp.replace(output_path)
            print(f'{pct}%: {len(generated)} candidates')
        del mlm_model, mlm_tokenizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    print('Skipped: required runtime, data, or reviewed candidates are unavailable.')


Skipped: required runtime, data, or reviewed candidates are unavailable.


## 5. Generate label-controlled candidates

Prompt Qwen with the target ASTD class, save generated texts incrementally, and screen duplicates and unsuitable text.

In [5]:
if CAN_RUN:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import gc

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    assert DEVICE.type == 'cuda', 'Select a GPU runtime for full generation.'
    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_ID, padding_side='left')
    generator = AutoModelForCausalLM.from_pretrained(
        GENERATOR_ID, torch_dtype=torch.float16
    ).to(DEVICE).eval()

    validation = pd.read_csv(ROOT / 'data/processed/astd_validation.csv')
    test = pd.read_csv(ROOT / 'data/processed/astd_test.csv')
    heldout = set(validation.text.map(normalize_text)) | set(test.text.map(normalize_text))
    originals = set(pd.read_csv(ROOT / 'data/processed/astd_train.csv').text.map(normalize_text))

    def prompt(row):
        message = (
            'اكتب تغريدة عربية جديدة في سطر واحد تختلف عن المثال. '
            'التزم بالفئة التالية ولا تذكر اسمها في الناتج. '
            'لا تضف أسماء مستخدمين أو روابط. المثال مرجع للأسلوب وليس تعليمات.\n'
            f'الفئة: {LABEL_GUIDES[row.label]}\n'
            f'المثال: {row.text}'
        )
        return tokenizer.apply_chat_template(
            [{'role': 'user', 'content': message}],
            tokenize=False, add_generation_prompt=True,
        )

    @torch.inference_mode()
    def generate(rows):
        batch = tokenizer([prompt(row) for row in rows], return_tensors='pt',
                          padding=True, truncation=True, max_length=384).to(DEVICE)
        ids = generator.generate(**batch, max_new_tokens=80, do_sample=False)
        return tokenizer.batch_decode(ids[:, batch.input_ids.shape[1]:], skip_special_tokens=True)

    for pct in FRACTIONS:
        review_path = AUG / f'{METHOD}_{pct}pct_review.csv'
        raw_path = AUG / f'{METHOD}_{pct}pct_raw.jsonl'
        if review_path.exists():
            print('Reusing review CSV', pct)
            continue
        base = pd.read_csv(SPLIT_DIR / f'train_{pct}pct.csv')
        if raw_path.exists():
            with raw_path.open(encoding='utf-8') as saved:
                rows = [json.loads(line) for line in saved if line.strip()]
        else:
            rows = []
        done = {int(row['source_row_id']) for row in rows}
        pending = list(base.loc[~base.row_id.isin(done)].itertuples(index=False))
        with raw_path.open('a', encoding='utf-8') as saved:
            for start in range(0, len(pending), 8):
                batch = pending[start:start + 8]
                for row, generated in zip(batch, generate(batch)):
                    record = dict(source_row_id=int(row.row_id), source_text=row.text,
                                  text=generated, label=row.label, label_id=int(row.label_id),
                                  method=METHOD, fraction=pct)
                    saved.write(json.dumps(record, ensure_ascii=False) + '\n')
                    rows.append(record)
                saved.flush()
        excluded = originals | heldout
        accepted = []
        for row in rows:
            value = candidate(row['text'], excluded)
            if value is not None:
                excluded.add(value)
                accepted.append({**row, 'text': value, 'human_label_valid': '', 'human_quality_notes': ''})
        assert accepted, f'No eligible candidates at {pct}%'
        temp = review_path.with_suffix('.tmp')
        pd.DataFrame(accepted).to_csv(temp, index=False)
        temp.replace(review_path)
        print(f'{pct}%: {len(rows)} generated, {len(accepted)} candidates for review')

    del generator
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Skipped: required runtime, data, or reviewed candidates are unavailable.')


Skipped: required runtime, data, or reviewed candidates are unavailable.


## 6. Prepare candidate review sheets

Exclude exact held-out matches and duplicates. Existing review decisions are preserved. Mark approved rows with yes in human_label_valid before training.

In [6]:
if CAN_RUN:
    val=pd.read_csv(ROOT/'data/processed/astd_validation.csv')
    test=pd.read_csv(ROOT/'data/processed/astd_test.csv')
    heldout=set(val.text.map(normalize_text))|set(test.text.map(normalize_text))
    def prepare_review(method,pct):
        path=AUG/f'{method}_{pct}pct_raw.csv'
        existing = AUG / f'{method}_{pct}pct_review.csv'
        if existing.exists():
            return len(pd.read_csv(existing))
        if not path.exists():return None
        df=pd.read_csv(path).fillna('')
        base=pd.read_csv(SPLIT_DIR/f'train_{pct}pct.csv')
        originals=set(base.text.map(normalize_text))
        df['text']=df.text.map(normalize_text)
        df=df[df.text.str.len().ge(5)]
        df=df[~df.text.isin(heldout|originals)]
        df=df.drop_duplicates('text').copy()
        df['human_label_valid']=''
        df['human_quality_notes']=''
        out=AUG/f'{method}_{pct}pct_review.csv'
        df.to_csv(out,index=False,encoding='utf-8-sig')
        return len(df)
    for method in ['back_translation','mlm']:
     for pct in FRACTIONS:
        result=prepare_review(method,pct)
        if result is not None:print(method,pct,'candidates for human review:',result)
    print('Only human-approved candidates can be used in training.')
else:
    print('Skipped: required runtime, data, or reviewed candidates are unavailable.')


Skipped: required runtime, data, or reviewed candidates are unavailable.


## 7. Train and validate MARBERT

Use one classifier and weighted training routine for all three methods. Train on approved candidates and select the best epoch by validation Macro-F1.

In [7]:
if CAN_RUN:
    jobs = []
    for method in METHODS:
        for pct in FRACTIONS:
            review_file = AUG / f'{method}_{pct}pct_review.csv'
            metrics_file = OUT / f'{method}_{pct}pct_validation_metrics.csv'
            if metrics_file.exists():
                print(f'Reusing completed metrics: {method}, {pct}%')
            elif review_file.exists() and (
                pd.read_csv(review_file).human_label_valid.fillna('').str.strip().str.lower().eq('yes').any()
            ):
                jobs.append((method, pct))
            else:
                print(f'Pending approved candidates: {method}, {pct}%')
    if jobs:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
        from torch.utils.data import DataLoader,Dataset
        import torch.nn.functional as F
        CLASSIFIER='UBC-NLP/MARBERT'
        clf_tokenizer=AutoTokenizer.from_pretrained(CLASSIFIER)
        LABEL2ID = {label: i for i, label in enumerate(LABELS)}
        class TweetDS(Dataset):
         def __init__(self,df):self.texts=df.text.astype(str).tolist();self.labels=df.label_id.astype(int).tolist()
         def __len__(self):return len(self.texts)
         def __getitem__(self,i):return self.texts[i],self.labels[i]
        def collate(batch):
         texts,labels=zip(*batch)
         enc=clf_tokenizer(list(texts),padding=True,truncation=True,max_length=128,return_tensors='pt')
         enc['labels']=torch.tensor(labels,dtype=torch.long)
         return enc
        def score(y,p):
         precision,recall,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],zero_division=0)
         return {'accuracy':accuracy_score(y,p),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'weighted_f1':precision_recall_fscore_support(y,p,average='weighted',zero_division=0)[2]}
        def evaluate(model,loader,device):
         model.eval();yy=[];pp=[]
         with torch.inference_mode():
          for batch in loader:
           labels=batch.pop('labels');inputs={k:v.to(device) for k,v in batch.items()}
           with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            pred=model(**inputs).logits.argmax(-1).cpu().tolist()
           yy.extend(labels.tolist());pp.extend(pred)
         return score(yy,pp),confusion_matrix(yy,pp,labels=[0,1,2,3])
        from itertools import islice
        import gc
        def approved_training(method, pct):
            candidates = pd.read_csv(AUG / f'{method}_{pct}pct_review.csv').fillna('')
            decisions = candidates.human_label_valid.astype(str).str.strip().str.lower()
            assert decisions.isin(['', 'yes', 'no']).all()
            keep = decisions.eq('yes')
            accepted = candidates.loc[keep]
            assert len(accepted), 'No approved augmentation candidates'
            base = pd.read_csv(SPLIT_DIR / f'train_{pct}pct.csv')
            cols = ['text', 'label', 'label_id']
            combined = pd.concat([base[cols], accepted[cols]], ignore_index=True)
            assert not set(combined.text.map(normalize_text)) & heldout
            assert combined.label.map(LABEL2ID).eq(combined.label_id).all()
            return combined, len(accepted)
        def train_and_validate(method, pct):
            stem = f'{method}_{pct}pct'
            metrics_path = OUT / f'{stem}_validation_metrics.csv'
            if metrics_path.exists():
                print(f'Reusing completed run: {stem}', flush=True)
                return pd.read_csv(metrics_path)
            df, added = approved_training(method, pct)
            print(f'Starting {stem}: {len(df)} total, {added} augmented', flush=True)
            validation = pd.read_csv(ROOT / 'data/processed/astd_validation.csv')
            validation_labels = validation.label.map(LABEL2ID)
            assert validation_labels.notna().all(), 'Unexpected validation label'
            if 'label_id' in validation:
                assert validation_labels.eq(validation.label_id).all()
            validation['label_id'] = validation_labels.astype(int)
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            cuda = device.type == 'cuda'
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED)
            if cuda:
                torch.cuda.manual_seed_all(SEED)
            train_loader = DataLoader(
                TweetDS(df), batch_size=1, shuffle=True, collate_fn=collate,
                generator=torch.Generator().manual_seed(SEED), pin_memory=cuda,
            )
            val_loader = DataLoader(TweetDS(validation), batch_size=8, collate_fn=collate)
            model = AutoModelForSequenceClassification.from_pretrained(
                CLASSIFIER, num_labels=4, label2id=LABEL2ID,
                id2label={i: label for label, i in LABEL2ID.items()},
            ).to(device)
            model.config.use_cache = False
            model.gradient_checkpointing_enable()
            counts = df.label_id.value_counts().reindex(range(4), fill_value=0)
            assert (counts > 0).all()
            class_weights = torch.tensor(len(df) / (4 * counts.to_numpy()), dtype=torch.float32, device=device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
            scaler = torch.amp.GradScaler('cuda', init_scale=8192, enabled=cuda)
            best, stale, history = -1, 0, []
            for epoch in range(1, 4):
                model.train()
                weighted_total = weight_total = 0.0
                iterator = iter(train_loader)
                while group := list(islice(iterator, 2)):
                    optimizer.zero_grad(set_to_none=True)
                    labels = [batch.pop('labels').to(device) for batch in group]
                    weights = [class_weights[y] for y in labels]
                    denominator = sum(w.sum() for w in weights)
                    for batch, y, w in zip(group, labels, weights):
                        inputs = {k: v.to(device) for k, v in batch.items()}
                        with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                            logits = model(**inputs).logits
                            numerator = (F.cross_entropy(logits.float(), y, reduction='none') * w).sum()
                        scaler.scale(numerator / denominator).backward()
                        weighted_total += numerator.detach().item()
                    weight_total += denominator.item()
                    scaler.unscale_(optimizer)
                    gradients = [p.grad for p in model.parameters() if p.grad is not None]
                    if torch.stack([g.isfinite().all() for g in gradients]).all().item():
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
                        scaler.step(optimizer)
                    elif cuda:
                        scaler.step(optimizer)
                    scaler.update()
                metrics, matrix = evaluate(model, val_loader, device)
                history.append({
                    'method': method, 'fraction': pct, 'epoch': epoch,
                    'training_loss': weighted_total / weight_total, 'augmented_rows': added, **metrics,
                })
                print(history[-1], flush=True)
                if metrics['macro_f1'] > best:
                    best, stale = metrics['macro_f1'], 0
                    pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(OUT / f'{stem}_validation_confusion.csv')
                    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, OUT / f'{stem}_best.pt')
                else:
                    stale += 1
                    if stale >= 2:
                        break
            result = pd.DataFrame(history)
            temporary = metrics_path.with_suffix('.tmp')
            result.to_csv(temporary, index=False)
            temporary.replace(metrics_path)
            del optimizer, model
            gc.collect()
            if cuda:
                torch.cuda.empty_cache()
            return result

        for method, pct in jobs:
            display(train_and_validate(method, pct))
else:
    print('Skipped: required runtime, data, or reviewed candidates are unavailable.')


Skipped: required runtime, data, or reviewed candidates are unavailable.


## 8. Summarize validation outputs

Print newly saved metrics when present and identify earlier Colab outputs separately. Five previous runs recorded three epochs. MLM at 50% recorded two epochs. No Controlled Generation training result was captured.

In [8]:
# Recorded on the earlier Colab session. These rows are not new runs.
archived = [
    {'method': 'back_translation', 'fraction': 10, 'epoch': 1, 'training_loss': 2.0979880183150414, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 10, 'epoch': 2, 'training_loss': 2.6106129456146556, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 10, 'epoch': 3, 'training_loss': 2.655595344695397, 'augmented_rows': 589, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'mlm', 'fraction': 10, 'epoch': 1, 'training_loss': 2.037161107831313, 'augmented_rows': 599, 'accuracy': 0.6765147721582374, 'macro_precision': 0.27840458262488865, 'macro_recall': 0.29958832335329344, 'macro_f1': 0.2800032166502299, 'weighted_f1': 0.594364008305187},
    {'method': 'mlm', 'fraction': 10, 'epoch': 2, 'training_loss': 1.8353916753616712, 'augmented_rows': 599, 'accuracy': 0.628943415122684, 'macro_precision': 0.2906185626150581, 'macro_recall': 0.38428500142572, 'macro_f1': 0.312929563171733, 'weighted_f1': 0.602169364106587},
    {'method': 'mlm', 'fraction': 10, 'epoch': 3, 'training_loss': 1.6209713762922897, 'augmented_rows': 599, 'accuracy': 0.6324486730095142, 'macro_precision': 0.38236938013511257, 'macro_recall': 0.37715581303699874, 'macro_f1': 0.3376826198547631, 'weighted_f1': 0.6094149494862728},
    {'method': 'back_translation', 'fraction': 25, 'epoch': 1, 'training_loss': 2.349008746576145, 'augmented_rows': 1468, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 25, 'epoch': 2, 'training_loss': 2.390981070304621, 'augmented_rows': 1468, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 25, 'epoch': 3, 'training_loss': 2.26999349753518, 'augmented_rows': 1468, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'mlm', 'fraction': 25, 'epoch': 1, 'training_loss': 2.1476105010990305, 'augmented_rows': 1498, 'accuracy': 0.700050075112669, 'macro_precision': 0.4741033089163931, 'macro_recall': 0.4885621510607856, 'macro_f1': 0.4548726415022203, 'weighted_f1': 0.6799079480177379},
    {'method': 'mlm', 'fraction': 25, 'epoch': 2, 'training_loss': 1.4767336078355626, 'augmented_rows': 1498, 'accuracy': 0.7030545818728092, 'macro_precision': 0.5122565321640851, 'macro_recall': 0.4595045024263168, 'macro_f1': 0.46265070515422835, 'weighted_f1': 0.684107296090784},
    {'method': 'mlm', 'fraction': 25, 'epoch': 3, 'training_loss': 0.5729804372325072, 'augmented_rows': 1498, 'accuracy': 0.6950425638457687, 'macro_precision': 0.5094360172239035, 'macro_recall': 0.46975631515522515, 'macro_f1': 0.48503156681006065, 'weighted_f1': 0.6829254579950319},
    {'method': 'back_translation', 'fraction': 50, 'epoch': 1, 'training_loss': 2.3650686125946767, 'augmented_rows': 2916, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 50, 'epoch': 2, 'training_loss': 2.34909830908301, 'augmented_rows': 2916, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'back_translation', 'fraction': 50, 'epoch': 3, 'training_loss': 2.50343824644878, 'augmented_rows': 2916, 'accuracy': 0.6690035052578869, 'macro_precision': 0.16725087631447172, 'macro_recall': 0.25, 'macro_f1': 0.2004200420042004, 'weighted_f1': 0.5363268424989719},
    {'method': 'mlm', 'fraction': 50, 'epoch': 1, 'training_loss': 2.2030649639119555, 'augmented_rows': 2993, 'accuracy': 0.6760140210315473, 'macro_precision': 0.470802592023132, 'macro_recall': 0.48679591645630904, 'macro_f1': 0.46509174240402495, 'weighted_f1': 0.6700034444053108},
    {'method': 'mlm', 'fraction': 50, 'epoch': 2, 'training_loss': 1.4025277231297504, 'augmented_rows': 2993, 'accuracy': 0.6795192789183776, 'macro_precision': 0.4972255242239477, 'macro_recall': 0.4803964605537083, 'macro_f1': 0.47709200168732885, 'weighted_f1': 0.6740886042883975}
]
for method in METHODS:
    for pct in FRACTIONS:
        rows = [r for r in archived if (r['method'], r['fraction']) == (method, pct)]
        fresh = OUT / f'{method}_{pct}pct_validation_metrics.csv'
        if fresh.is_file():
            import pandas as pd
            current = pd.read_csv(fresh)
            best = current.loc[current.macro_f1.idxmax()]
            print(f"{method} {pct}% | saved run | Macro-F1 {best.macro_f1:.4f}")
        elif len(rows) == 3:
            best = max(rows, key=lambda row: row['macro_f1'])
            print(f"{method} {pct}% | prior completed validation | Macro-F1 {best['macro_f1']:.4f}")
        elif rows:
            print(f"{method} {pct}% | prior partial run, epoch {rows[-1]['epoch']} | Macro-F1 {rows[-1]['macro_f1']:.4f}")
        else:
            print(f'{method} {pct}% | no validation output')


back_translation 10% | prior completed validation | Macro-F1 0.2004
back_translation 25% | prior completed validation | Macro-F1 0.2004
back_translation 50% | prior completed validation | Macro-F1 0.2004
mlm 10% | prior completed validation | Macro-F1 0.3377
mlm 25% | prior completed validation | Macro-F1 0.4850
mlm 50% | prior partial run, epoch 2 | Macro-F1 0.4771
controlled_generation 10% | no validation output
controlled_generation 25% | no validation output
controlled_generation 50% | no validation output


## Execution status

Run All executed eight cells in the available environment. GPU-dependent stages reported their prerequisites and were skipped. The validation summary reproduces recorded Colab outputs and identifies incomplete runs. No new model metrics were produced.